# NOTEBOOK 06 — MACHINE LEARNING & 20-YEAR FORECAST

### Mục tiêu
1. Đọc `climate_country_features` từ PostgreSQL.
2. Dùng **time-based split**, không random shuffle.
3. So sánh Seasonal Climatology Baseline, Ridge, Random Forest, HistGradientBoosting.
4. Tối ưu hyperparameter bằng date-based cross-validation.
5. Đánh giá MAE, RMSE, R², Actual vs Predicted và Residual.
6. Ablation để kiểm tra giá trị của auxiliary features.
7. Chọn model phù hợp cho **20-year extrapolation**.
8. Dự báo 240 tháng cho 5 quốc gia.
9. Lưu model, metrics và forecast cho Notebook 07.


## I. Setup và đọc Feature Table

In [ ]:
from pathlib import Path
import os

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"
APP_DIR = PROJECT_ROOT / "app"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url_object = URL.create(
    "postgresql",
    username="postgres",
    password="123456",
    host="localhost",
    port=5432,
    database="climate_change_db1",
)
engine = create_engine(url_object, pool_pre_ping=True)

with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

registry = json.loads(
    (ARTIFACTS / 'feature_registry.json').read_text(encoding='utf-8')
)

BASE_FEATURES = registry.get('base_features', registry.get('time_features', []))
COUNTRY_TREND_FEATURES = registry.get('country_trend_features', registry.get('target_history_features', []))
AUX_FEATURES = registry.get('aux_features', [])
FEATURES = registry.get('features', [])
FORECAST_SAFE_FEATURES = registry.get('forecast_safe_features', FEATURES)
HISTORICAL_ONLY_FEATURES = registry.get('historical_only_features', [])

TARGET = 'average_temperature_observed'

df = pd.read_sql(
    text('SELECT * FROM climate_country_features ORDER BY dt, country'),
    engine,
    parse_dates=['dt']
)

print('Shape:', df.shape)
print('Feature count:', len(FEATURES))

## II. Modeling Dataset

Ground truth chỉ gồm nhiệt độ quan sát thật.
Các dòng target nội suy không được dùng làm nhãn đánh giá.

In [ ]:
model_df = (
    df[
        (df['target_is_observed'] == 1)
        & df[TARGET].notna()
    ]
    .sort_values(['dt', 'country'])
    .reset_index(drop=True)
)

print('Model rows:', len(model_df))
print('Date range:', model_df['dt'].min(), '→', model_df['dt'].max())

## III. Time-based Train/Test Split

- Train: trước `2004-01-01`
- Test: từ `2004-01-01` đến hết lịch sử

Mốc này khớp Notebook 05, nơi climatology/trend từ bảng phụ chỉ dùng dữ liệu ≤ 2003.

In [ ]:
TEST_START = pd.Timestamp('2004-01-01')

train_df = model_df[model_df['dt'] < TEST_START].copy()
test_df = model_df[model_df['dt'] >= TEST_START].copy()

print('Train:', train_df['dt'].min(), '→', train_df['dt'].max(), len(train_df))
print('Test :', test_df['dt'].min(), '→', test_df['dt'].max(), len(test_df))

assert train_df['dt'].max() < test_df['dt'].min()

## IV. Baseline — Country × Month Climatology

Baseline dự đoán:

> nhiệt độ tháng tương lai = trung bình lịch sử của **cùng Country + Month** trong train.

Đây là baseline mạnh cho dữ liệu có seasonality.

In [ ]:
def evaluate(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred)
    }

train_climatology = (
    train_df.groupby(['country', 'month'])[TARGET]
            .mean()
            .rename('baseline_prediction')
            .reset_index()
)

baseline_test = test_df.merge(
    train_climatology,
    on=['country', 'month'],
    how='left'
)

baseline_metrics = evaluate(
    baseline_test[TARGET],
    baseline_test['baseline_prediction']
)

display(pd.DataFrame([{'Model': 'Climatology Baseline', **baseline_metrics}]))

## V. Preprocessing

- `country`: One-Hot Encoding
- numeric features: median imputation
- Ridge: thêm StandardScaler
- Tree models: giữ numeric scale gốc

Không fit imputer/scaler trên test.

In [ ]:
try:
    ohe_ridge = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )
    ohe_tree = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )
except TypeError:
    ohe_ridge = OneHotEncoder(
        handle_unknown='ignore',
        sparse=False
    )
    ohe_tree = OneHotEncoder(
        handle_unknown='ignore',
        sparse=False
    )

ridge_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

tree_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

ridge_prep = ColumnTransformer([
    ('country', ohe_ridge, ['country']),
    ('num', ridge_num, FEATURES)
])

tree_prep = ColumnTransformer([
    ('country', ohe_tree, ['country']),
    ('num', tree_num, FEATURES)
])

X_train = train_df[['country'] + FEATURES]
y_train = train_df[TARGET]

X_test = test_df[['country'] + FEATURES]
y_test = test_df[TARGET]

## VI. Date-based Cross Validation

Không dùng random K-fold.
Mỗi fold validation là một block thời gian nằm sau phần train của fold đó.

In [ ]:
def make_date_folds(frame, n_splits=3, val_months=48):
    unique_dates = np.array(sorted(frame['dt'].unique()))
    folds = []

    min_train_months = 120
    max_possible = max(
        12,
        (len(unique_dates) - min_train_months) // n_splits
    )
    val_months = min(val_months, max_possible)

    for split in range(n_splits):
        val_end = (
            len(unique_dates)
            - (n_splits - split - 1) * val_months
        )
        val_start = val_end - val_months

        train_dates = unique_dates[:val_start]
        valid_dates = unique_dates[val_start:val_end]

        tr_idx = np.flatnonzero(
            frame['dt'].isin(train_dates).to_numpy()
        )
        va_idx = np.flatnonzero(
            frame['dt'].isin(valid_dates).to_numpy()
        )

        if len(tr_idx) and len(va_idx):
            folds.append((tr_idx, va_idx))

    return folds

cv_folds = make_date_folds(
    train_df,
    n_splits=3,
    val_months=48
)

print([(len(tr), len(va)) for tr, va in cv_folds])

## VII. Ridge Regression + GridSearchCV

In [ ]:
ridge_pipe = Pipeline([
    ('prep', ridge_prep),
    ('model', Ridge())
])

ridge_search = GridSearchCV(
    ridge_pipe,
    param_grid={
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    cv=cv_folds,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)
ridge_best = ridge_search.best_estimator_
pred_ridge = ridge_best.predict(X_test)

print('Best Ridge params:', ridge_search.best_params_)
print(f'Best Ridge CV RMSE: {-ridge_search.best_score_:.4f}')


## VIII. Random Forest + GridSearchCV

In [ ]:
rf_pipe = Pipeline([
    ('prep', tree_prep),
    ('model', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

rf_search = GridSearchCV(
    rf_pipe,
    param_grid={
        'model__n_estimators': [250, 500],
        'model__max_depth': [12, 20, None],
        'model__min_samples_leaf': [2, 5]
    },
    cv=cv_folds,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
pred_rf = rf_best.predict(X_test)

print('Best RF params:', rf_search.best_params_)
print(f'Best RF CV RMSE: {-rf_search.best_score_:.4f}')


## IX. HistGradientBoosting + GridSearchCV

In [ ]:
hgb_pipe = Pipeline([
    ('prep', tree_prep),
    ('model', HistGradientBoostingRegressor(
        random_state=42
    ))
])

hgb_search = GridSearchCV(
    hgb_pipe,
    param_grid={
        'model__learning_rate': [0.03, 0.06, 0.1],
        'model__max_leaf_nodes': [15, 31],
        'model__l2_regularization': [0.0, 1.0]
    },
    cv=cv_folds,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

hgb_search.fit(X_train, y_train)
hgb_best = hgb_search.best_estimator_
pred_hgb = hgb_best.predict(X_test)

print('Best HGB params:', hgb_search.best_params_)
print(f'Best HGB CV RMSE: {-hgb_search.best_score_:.4f}')


## X. So sánh Models

In [ ]:
results = pd.DataFrame([
    {'Model': 'Climatology Baseline', **baseline_metrics},
    {'Model': 'Ridge', **evaluate(y_test, pred_ridge)},
    {'Model': 'Random Forest', **evaluate(y_test, pred_rf)},
    {'Model': 'HistGradientBoosting', **evaluate(y_test, pred_hgb)}
]).sort_values('RMSE').reset_index(drop=True)

display(results)

### Metric theo từng quốc gia

In [ ]:
prediction_table = test_df[
    ['dt', 'country', TARGET]
].copy()

prediction_table['Ridge'] = pred_ridge
prediction_table['Random Forest'] = pred_rf
prediction_table['HistGradientBoosting'] = pred_hgb

country_metrics = []

for country, group in prediction_table.groupby('country'):
    for model_name in [
        'Ridge',
        'Random Forest',
        'HistGradientBoosting'
    ]:
        country_metrics.append({
            'country': country,
            'Model': model_name,
            **evaluate(group[TARGET], group[model_name])
        })

country_metrics_df = pd.DataFrame(country_metrics)
display(
    country_metrics_df.sort_values(['country', 'RMSE'])
)

## XI. Actual vs Predicted và Residual

Hiển thị Ridge vì đây là ứng viên cho long-horizon extrapolation.

In [ ]:
ridge_plot = test_df[['dt', 'country', TARGET]].copy()
ridge_plot['prediction'] = pred_ridge
ridge_plot['residual'] = (
    ridge_plot[TARGET] - ridge_plot['prediction']
)

fig, ax = plt.subplots(figsize=(12, 5))
for country, group in ridge_plot.groupby('country'):
    ax.plot(
        group['dt'],
        group[TARGET],
        alpha=0.45,
        label=f'{country} actual'
    )
    ax.plot(
        group['dt'],
        group['prediction'],
        linestyle='--',
        alpha=0.8,
        label=f'{country} predicted'
    )

ax.set_title('Ridge — Actual vs Predicted')
ax.set_ylabel('°C')
ax.legend(ncol=2, fontsize=8)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(
    ridge_plot['prediction'],
    ridge_plot['residual'],
    alpha=0.35
)
ax.axhline(0, linestyle='--')
ax.set_xlabel('Prediction')
ax.set_ylabel('Residual')
ax.set_title('Ridge residual plot')
plt.show()

## XII. Ablation — auxiliary features có thực sự giúp?

So Ridge với:
1. Base Features
2. Full feature set (Base Features + Aux Features)

Đây là bằng chứng trực tiếp cho giá trị Feature Engineering.

In [ ]:
def build_ridge(features, alpha):
    try:
        encoder = OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown='ignore',
            sparse=False
        )

    prep = ColumnTransformer([
        ('country', encoder, ['country']),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), features)
    ])

    return Pipeline([
        ('prep', prep),
        ('model', Ridge(alpha=alpha))
    ])

reduced_features = BASE_FEATURES + AUX_FEATURES
ridge_alpha = ridge_search.best_params_['model__alpha']

ridge_reduced = build_ridge(
    reduced_features,
    ridge_alpha
)
ridge_reduced.fit(
    train_df[['country'] + reduced_features],
    y_train
)

pred_reduced = ridge_reduced.predict(
    test_df[['country'] + reduced_features]
)

ablation = pd.DataFrame([
    {
        'FeatureSet': 'Base Features + Auxiliary Features',
        **evaluate(y_test, pred_reduced)
    },
    {
        'FeatureSet': 'Full Features',
        **evaluate(y_test, pred_ridge)
    }
])

display(ablation)

## XIII. Model selection cho 20-year forecast

Tree models có thể đạt RMSE thấp trên historical holdout nhưng **không extrapolate linear time trend ngoài miền training** một cách tự nhiên.

Với bài toán 20 năm, production model dùng **Ridge** vì:
- có `time_idx`;
- có slope riêng;
- có Fourier seasonality;
- toàn bộ feature tính được cho future date.

Notebook vẫn báo `short_term_best_model` theo RMSE để minh bạch.

In [ ]:
short_term_best_model = results.iloc[0]['Model']
PRODUCTION_MODEL_NAME = 'Ridge'
production_model = ridge_best

print('Best historical holdout RMSE:', short_term_best_model)
print('Production 20-year model:', PRODUCTION_MODEL_NAME)

## XIV. Retrain production model trên toàn bộ observed history

Sau khi model/hyperparameter đã được chọn bằng test, fit lại Ridge trên toàn bộ dữ liệu quan sát trước khi forecast tương lai.

In [ ]:
X_all = model_df[['country'] + FEATURES]
y_all = model_df[TARGET]

production_model.fit(X_all, y_all)

## XV. Tạo feature cho 240 tháng tương lai

Recursive prediction required for `forecast_safe_features` if they contain target lags.

In [ ]:
lookup = pd.read_sql(
    text('SELECT * FROM climate_feature_lookup ORDER BY country, month'),
    engine
)

historical_end = df['dt'].max()

def build_future_rows(country, start_date, periods=240):
    dates = pd.date_range(
        start_date,
        periods=periods,
        freq='MS'
    )

    future = pd.DataFrame({
        'dt': dates,
        'country': country
    })

    future['year'] = future['dt'].dt.year
    future['month'] = future['dt'].dt.month
    future['quarter'] = future['dt'].dt.quarter
    future['time_idx'] = (
        (future['year'] - 1850) * 12
        + (future['month'] - 1)
    )
    future['month_sin'] = np.sin(
        2 * np.pi * future['month'] / 12
    )
    future['month_cos'] = np.cos(
        2 * np.pi * future['month'] / 12
    )

    country_lookup = lookup[
        lookup['country'] == country
    ].copy()

    future = future.merge(
        country_lookup,
        on=['country', 'month'],
        how='left'
    )
    
    # Initialize recursive target features with NA
    for f in FORECAST_SAFE_FEATURES:
        if f not in future.columns:
            future[f] = np.nan

    return future

future_parts = []

TARGET_COUNTRIES = [
    'Australia',
    'France',
    'Japan',
    'United Kingdom',
    'United States'
]

# Forecast for 5 countries using recursive predictions
for country in TARGET_COUNTRIES:
    start = historical_end + pd.offsets.MonthBegin(1)
    country_future = build_future_rows(
        country,
        start_date=start,
        periods=240
    )
    
    # Get last known history for this country
    history = model_df[model_df['country'] == country].sort_values('dt').copy()
    
    predictions = []
    
    for i in range(len(country_future)):
        current_row = country_future.iloc[[i]].copy()
        
        # Calculate lag features from history
        for lag in [1, 3, 6, 12, 24]:
            col = f'temp_lag_{lag}'
            if col in FORECAST_SAFE_FEATURES:
                if len(history) >= lag:
                    current_row[col] = history['average_temperature_filled'].iloc[-lag]
                else:
                    current_row[col] = np.nan
                    
        # Calculate rolling features
        if 'temp_roll_mean_3' in FORECAST_SAFE_FEATURES and len(history) >= 3:
            current_row['temp_roll_mean_3'] = history['average_temperature_filled'].iloc[-3:].mean()
        if 'temp_roll_mean_6' in FORECAST_SAFE_FEATURES and len(history) >= 6:
            current_row['temp_roll_mean_6'] = history['average_temperature_filled'].iloc[-6:].mean()
        if 'temp_roll_mean_12' in FORECAST_SAFE_FEATURES and len(history) >= 12:
            current_row['temp_roll_mean_12'] = history['average_temperature_filled'].iloc[-12:].mean()
        if 'temp_roll_std_12' in FORECAST_SAFE_FEATURES and len(history) >= 12:
            current_row['temp_roll_std_12'] = history['average_temperature_filled'].iloc[-12:].std()

        # Predict
        pred = production_model.predict(current_row[['country'] + FEATURES])[0]
        predictions.append(pred)
        
        # Add to history for next iteration
        next_history_row = current_row.copy()
        next_history_row['average_temperature_filled'] = pred
        history = pd.concat([history, next_history_row], ignore_index=True)
        
    country_future['predicted_temperature'] = predictions
    future_parts.append(country_future)

future = pd.concat(
    future_parts,
    ignore_index=True
)

print('Forecast:', future['dt'].min(), '→', future['dt'].max())
display(future.head())

## XVI. Khoảng dự báo thực nghiệm

Dùng residual standard deviation của Ridge trên test theo từng quốc gia.
Đây là **empirical prediction band**, không phải climate-model confidence interval vật lý.

In [ ]:
residual_scale = (
    ridge_plot.groupby('country')['residual']
              .std()
              .rename('residual_std')
              .reset_index()
)

future = future.merge(
    residual_scale,
    on='country',
    how='left'
)

future['prediction_lower'] = (
    future['predicted_temperature']
    - 1.96 * future['residual_std']
)
future['prediction_upper'] = (
    future['predicted_temperature']
    + 1.96 * future['residual_std']
)

## XVII. Tổng hợp warming trend dự báo

So mean 5 năm đầu với mean 5 năm cuối của horizon 20 năm.

In [ ]:
forecast_summary_rows = []

for country, group in future.groupby('country'):
    g = group.sort_values('dt')
    first_5y = g.head(60)['predicted_temperature'].mean()
    last_5y = g.tail(60)['predicted_temperature'].mean()

    forecast_summary_rows.append({
        'country': country,
        'mean_first_5y_C': first_5y,
        'mean_last_5y_C': last_5y,
        'change_last_vs_first_5y_C': last_5y - first_5y
    })

forecast_summary = pd.DataFrame(
    forecast_summary_rows
)
display(forecast_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for country, group in future.groupby('country'):
    ax.plot(
        group['dt'],
        group['predicted_temperature'],
        label=country
    )

ax.set_title('20-year monthly temperature forecast')
ax.set_ylabel('Predicted average temperature (°C)')
ax.legend()
plt.show()

## XVIII. Lưu model, metrics và forecast

In [ ]:
MODEL_PATH = ARTIFACTS / 'climate_ridge_20y.joblib'
joblib.dump(production_model, MODEL_PATH)

metadata = {
    'model_name': PRODUCTION_MODEL_NAME,
    'short_term_best_model': short_term_best_model,
    'target': TARGET,
    'features': FEATURES,
    'base_features': BASE_FEATURES,
    'country_trend_features': COUNTRY_TREND_FEATURES,
    'aux_features': AUX_FEATURES,
    'forecast_safe_features': FORECAST_SAFE_FEATURES,
    'historical_only_features': HISTORICAL_ONLY_FEATURES,
    'test_start': str(TEST_START.date()),
    'historical_end': str(historical_end.date()),
    'forecast_horizon_months': 240,
    'metrics': json.loads(results.to_json(orient='records')),
    'ablation': json.loads(ablation.to_json(orient='records')),
    'warning': (
        'Statistical temperature forecast; '
        'not a physical climate-scenario projection.'
    )
}

(ARTIFACTS / 'model_metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

forecast_cols = [
    'dt',
    'country',
    'predicted_temperature',
    'prediction_lower',
    'prediction_upper'
]

future[forecast_cols].to_sql(
    'climate_forecast_20y',
    engine,
    if_exists='replace',
    index=False,
    chunksize=5000,
    method='multi'
)

future[forecast_cols].to_csv(
    DATA_PROCESSED / 'climate_forecast_20y.csv',
    index=False
)

print('Saved model:', MODEL_PATH)
print('Saved climate_forecast_20y.')

## XIX. Kết luận Notebook 06

Phải trả lời:
1. Baseline MAE/RMSE/R² là bao nhiêu?
2. Model nào có historical holdout RMSE thấp nhất?
3. Auxiliary features cải thiện hay làm xấu Ridge?
4. Vì sao Ridge được chọn cho horizon 20 năm?
5. Dự báo 5 quốc gia thay đổi bao nhiêu giữa 5 năm đầu và 5 năm cuối?

### Cách diễn giải đúng
> “Mô hình thống kê dự báo xu hướng nhiệt độ tiếp tục thay đổi theo pattern và trend lịch sử.”

Không viết:
> “Mô hình chứng minh nguyên nhân biến đổi khí hậu.”